# 不使用@tool的方式定义工具

## 1、举例

In [1]:
# 1、模型的初始化
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

# 2、声明一个函数（工具）
def get_weather(city : str):
    return f"{city}天气晴朗~~"

# 3、将函数绑定在模型上
model_with_tools = model.bind_tools([get_weather])

# 4、调用模型
response = model_with_tools.invoke("北京的天气怎么样")
rprint(response)

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 18,
            'prompt_tokens': 121,
            'total_tokens': 139,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
            'latency_checkpoint': {
                'engine_tbt_ms': 3,
                'engine_ttft_ms': 37,
                'engine_ttlt_ms': 97,
                'pre_inference_ms': 85,
                'service_tbt_ms': 3,
                'service_ttft_ms': 270,
                'service_ttlt_ms': 318,
                'total_duration_ms': 240,
                'user_visible_ttft_ms': 185
            }
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5.4-mini-2026-03-17',
        'system_fingerprint': None,
        'id': 'chatcmpl-DjOt2vKA0RVOwxjckkxKArISUdEDj',
        'service_tier': 'default',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019e5f26-1e58-7641-a2a0-cc3dbca477a1-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '北京'},
            'id': 'call_sbsWIOPicbOqiBTR3uKaeaoS',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 121,
        'output_tokens': 18,
        'total_tokens': 139,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

## 2、工具描述的各部分详解

## 2.1 了解convert_to_openai_tool

执行`model.bind_tools([get_weather])`，底层最终会调用`convert_to_openai_tool`生成工具描述。所以我们可以直接调用后者查看解析后的工具描述。

In [3]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.2 description说明

In [5]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    """
    查询城市的天气
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.3 参数说明

In [10]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    """
    查询城市的天气

    Args:
        city : 具体的城市

    Returns:
        返回城市的天气
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {'city': {'description': '具体的城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 2.4 参数类型说明

举例1：正确的

In [12]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city):
    """
    查询城市的天气
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {'properties': {'city': {}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2：如下的代码运行会报错

要求：如果在docstring中声明了参数的描述，则必须在函数声明处指明参数的类型

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city):
    """
    查询城市的天气

    Args:
        city : 具体的城市
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

## 2.5 参数默认值说明

一旦参数设置的默认值，则打印的结果中的required字段中就不再包含此参数。

举例1：

In [14]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str = "beijing"):
    """
    查询城市的天气

    Args:
        city : 具体的城市
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {'city': {'default': 'beijing', 'description': '具体的城市', 'type': 'string'}},
            'type': 'object'
        }
    }
}

举例2：

In [15]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(dt:str ,city : str = "beijing"):
    """
    查询城市的天气

    Args:
        city : 具体的城市
        dt : 时间
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {
                'dt': {'description': '时间', 'type': 'string'},
                'city': {'default': 'beijing', 'description': '具体的城市', 'type': 'string'}
            },
            'required': ['dt'],
            'type': 'object'
        }
    }
}